![image](car.jpeg)

**Car-ing is sharing**, an auto dealership company for car sales and rental, is taking their services to the next level thanks to **Large Language Models (LLMs)**.

As their newly recruited AI and NLP developer, you've been asked to prototype a chatbot app with multiple functionalities that not only assist customers but also provide support to human agents in the company.

The solution should receive textual prompts and use a variety of pre-trained Hugging Face LLMs to respond to a series of tasks, e.g. classifying the sentiment in a car’s text review, answering a customer question, summarizing or translating text, etc.


In [44]:
# Import necessary packages
import pandas as pd
import torch

from transformers import logging
logging.set_verbosity(logging.WARNING)

In [45]:
# Start your code here!

from transformers import pipeline
import evaluate

car_reviews = pd.read_csv("data/car_reviews.csv", sep=';')

In [46]:
# Task 1

print(car_reviews.columns)

car_reviews.columns = car_reviews.columns.str.strip().str.lower()

reviews_list = car_reviews["review"].tolist()
real_labels = car_reviews["class"].tolist()

classifier = pipeline(task="sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
predicted_labels = classifier(reviews_list)

predictions = [1 if item["label"] == "POSITIVE" else 0 for item in predicted_labels]
references = [1 if label == "POSITIVE" else 0 for label in real_labels]

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

accuracy_result = accuracy_metric.compute(
    predictions=predictions,
    references=references)['accuracy']
f1_result = f1_metric.compute(
    predictions=predictions,
    references=references)['f1']
print("accuracy: ", accuracy_result)
print("f1: ", f1_result)

Index(['Review', 'Class'], dtype='object')


Device set to use cpu


accuracy:  0.8
f1:  0.8571428571428571


In [47]:
# Task 2

first_review = car_reviews['review'].iloc[0]
first_two_sentences = ". ".join(first_review.split(".")[:2]).strip() + "."

translator = pipeline(task="translation", model = "Helsinki-NLP/opus-mt-en-es")
translate_output = translator(first_two_sentences)
translated_review = translate_output[0]["translation_text"] 

import os

reference_file = "data/reference_translations.txt"
if os.path.exists(reference_file):
    with open(reference_file, "r") as f:
        reference_translation = f.read().strip()
else:
    print(f"Warning: '{reference_file}' not found. Using the original review as reference.")
    reference_translation = first_two_sentences

bleu_metric = evaluate.load("bleu")
bleu_score = bleu_metric.compute(
    predictions = [translated_review],
    references = [[reference_translation]]
)
print("BLEU score:", bleu_score)

Device set to use cpu


BLEU score: {'bleu': 0.3140322081976493, 'precisions': [0.9090909090909091, 0.8571428571428571, 0.75, 0.631578947368421], 'brevity_penalty': 0.40289032152913307, 'length_ratio': 0.5238095238095238, 'translation_length': 22, 'reference_length': 42}


In [48]:
# Task 3

question = "What did he like about the brand?"
context = car_reviews["review"].iloc[1]
qa_pipeline = pipeline(task="question-answering", model="deepset/minilm-uncased-squad2")
qa_result = qa_pipeline(question=question, context=context)
answer = qa_result["answer"]

Some weights of the model checkpoint at deepset/minilm-uncased-squad2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


In [49]:
# Task 4

last_review = car_reviews["review"].iloc[-1]

summarizer = pipeline(task="summarization", model = "sshleifer/distilbart-cnn-12-6")

summary_output = summarizer(last_review, min_length=50, max_length=55)
summarized_text = summary_output[0]["summary_text"]

toxicity = evaluate.load("toxicity")
regard = evaluate.load("regard")

toxicity_result = toxicity.compute(predictions = [summarized_text], aggregation = "maximum")
regard_result = regard.compute(data = [summarized_text])

print("Max toxicity:", toxicity_result["max_toxicity"])
print("Regard scores:", regard_result["regard"])




Device set to use cpu
Device set to use cpu
Device set to use cpu


Max toxicity: 0.00013859833416063339
Regard scores: [[{'label': 'positive', 'score': 0.7860167622566223}, {'label': 'other', 'score': 0.09952664375305176}, {'label': 'neutral', 'score': 0.08915404230356216}, {'label': 'negative', 'score': 0.025302601978182793}]]
